# Embedding Similarity

Runs either a within-record prompt comparison or an exact global edit-retrieval experiment with configurable frozen sentence-embedding models. Results are separated by experiment mode and effective inference precision.

## Environment

In [ ]:
from contextlib import redirect_stderr
import gc
import hashlib
from io import StringIO
import json
import sys
from pathlib import Path
import unicodedata
from typing import Any

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, load_from_disk
from IPython.display import display
from tqdm.auto import tqdm


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "notebook").exists():
            return path
    raise RuntimeError(f"Could not find project root from {start}")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_REPO = "Savassan/KnowledgeSurgeon"
DATASET_CONFIG = "similarity"
DATASET_SPLIT = "data"
LOCAL_DATASET_PATH = PROJECT_ROOT / "data" / "datasets" / "uniedit" / "final"
OUTPUT_DIR = PROJECT_ROOT / "data" / "similarity" / "embedding"
REPRESENTATION_TYPE = "normalized_dense_sentence_embedding"

EMBEDDING_MODEL_SPECS = {
    "all-mpnet-base-v2": {
        "repo": "sentence-transformers/all-mpnet-base-v2",
        "backend": "sentence_transformers",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
    "all-MiniLM-L6-v2": {
        "repo": "sentence-transformers/all-MiniLM-L6-v2",
        "backend": "sentence_transformers",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
    "all-MiniLM-L12-v2": {
        "repo": "sentence-transformers/all-MiniLM-L12-v2",
        "backend": "sentence_transformers",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
    "e5-base-v2": {
        "repo": "intfloat/e5-base-v2",
        "backend": "sentence_transformers",
        "prefixes": {
            "symmetric": "query: ",
            "query": "query: ",
            "passage": "passage: ",
        },
    },
    "e5-large-v2": {
        "repo": "intfloat/e5-large-v2",
        "backend": "sentence_transformers",
        "prefixes": {
            "symmetric": "query: ",
            "query": "query: ",
            "passage": "passage: ",
        },
    },
    "bge-base-en-v1.5": {
        "repo": "BAAI/bge-base-en-v1.5",
        "backend": "sentence_transformers",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
    "bge-large-en-v1.5": {
        "repo": "BAAI/bge-large-en-v1.5",
        "backend": "sentence_transformers",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
    "bge-m3": {
        "repo": "BAAI/bge-m3",
        "backend": "flag_embedding",
        "prefixes": {"symmetric": "", "query": "", "passage": ""},
    },
}


def resolve_embedding_model_keys(model_keys: list[str]) -> list[str]:
    selected = list(dict.fromkeys(model_keys))
    if not selected:
        raise ValueError("embedding_model_keys must contain at least one model.")
    unknown = [key for key in selected if key not in EMBEDDING_MODEL_SPECS]
    if unknown:
        raise ValueError(f"Unknown embedding model keys: {unknown}")
    return selected


def validate_experiment_mode(experiment_mode: str) -> str:
    if experiment_mode not in {"local", "global"}:
        raise ValueError("experiment_mode must be 'local' or 'global'.")
    return experiment_mode


def similarity_output_path(
    model_key: str,
    experiment_mode: str,
    record_limit: int | None,
) -> Path:
    validate_experiment_mode(experiment_mode)
    if record_limit is not None and int(record_limit) <= 0:
        raise ValueError("record_limit must be positive when set.")
    output_stem = model_key if record_limit is None else f"{model_key}.limit-{int(record_limit)}"
    return OUTPUT_DIR / experiment_mode / embedding_precision_mode() / f"{output_stem}.json"


print(f"local_dataset={LOCAL_DATASET_PATH}")
print(f"output_dir={OUTPUT_DIR}")

## Embedding Backends

In [ ]:
def embedding_device() -> str:
    return f"cuda:{EMBEDDING_GPU_ID}" if torch.cuda.is_available() else "cpu"


def embedding_compute_dtype() -> str:
    return "float16" if EMBEDDING_USE_FP16 and torch.cuda.is_available() else "float32"


def embedding_precision_mode() -> str:
    return "fp16" if embedding_compute_dtype() == "float16" else "fp32"


def load_embedding_model(model_key: str):
    spec = EMBEDDING_MODEL_SPECS[model_key]
    device = embedding_device()
    if spec["backend"] == "sentence_transformers":
        from sentence_transformers import SentenceTransformer

        model = SentenceTransformer(spec["repo"], device=device)
        model.max_seq_length = min(int(model.max_seq_length), EMBEDDING_MAX_LENGTH)
        if embedding_compute_dtype() == "float16":
            model.half()
    elif spec["backend"] == "flag_embedding":
        from FlagEmbedding import BGEM3FlagModel

        model = BGEM3FlagModel(
            spec["repo"],
            use_fp16=embedding_compute_dtype() == "float16",
            devices=device,
        )
    else:
        raise ValueError(f"Unsupported embedding backend: {spec['backend']}")
    print(
        f"loaded_embedding_model={model_key} repo={spec['repo']} "
        f"backend={spec['backend']} device={device} "
        f"compute_dtype={embedding_compute_dtype()}"
    )
    return model


def embedding_effective_max_length(model, model_key: str) -> int:
    if EMBEDDING_MODEL_SPECS[model_key]["backend"] == "sentence_transformers":
        return int(model.max_seq_length)
    return EMBEDDING_MAX_LENGTH


def format_embedding_prompts(
    model_key: str,
    prompts: list[str],
    prompt_role: str,
) -> list[str]:
    prefixes = EMBEDDING_MODEL_SPECS[model_key]["prefixes"]
    if prompt_role not in prefixes:
        raise ValueError(f"Unsupported prompt role: {prompt_role}")
    prefix = prefixes[prompt_role]
    return [f"{prefix}{prompt}" for prompt in prompts]


def encode_flag_embedding_prompts(model, prompts: list[str]) -> dict[str, Any]:
    with redirect_stderr(StringIO()):
        return model.encode(
            prompts,
            batch_size=EMBEDDING_BATCH_SIZE,
            max_length=EMBEDDING_MAX_LENGTH,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )


def encode_prompts(
    model,
    model_key: str,
    prompts: list[str],
    prompt_role: str,
) -> np.ndarray:
    if not prompts:
        raise ValueError("Cannot encode an empty prompt list.")
    spec = EMBEDDING_MODEL_SPECS[model_key]
    formatted_prompts = format_embedding_prompts(model_key, prompts, prompt_role)
    if spec["backend"] == "sentence_transformers":
        embeddings = model.encode(
            formatted_prompts,
            batch_size=EMBEDDING_BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
    else:
        encoded = encode_flag_embedding_prompts(model, formatted_prompts)
        embeddings = encoded["dense_vecs"]

    embeddings = np.asarray(embeddings, dtype=np.float32)
    norms = np.linalg.norm(embeddings, axis=-1, keepdims=True)
    embeddings = embeddings / np.clip(norms, 1e-12, None)
    if embeddings.ndim != 2 or embeddings.shape[0] != len(prompts):
        raise ValueError(
            f"Unexpected embedding shape for {model_key}: {embeddings.shape}; "
            f"expected ({len(prompts)}, hidden_size)."
        )
    return embeddings


def move_embedding_model_to_cpu(model, model_key: str) -> None:
    backend = EMBEDDING_MODEL_SPECS[model_key]["backend"]
    try:
        if backend == "sentence_transformers":
            model.to("cpu")
        else:
            underlying_model = getattr(model, "model", None)
            if underlying_model is not None and hasattr(underlying_model, "to"):
                underlying_model.to("cpu")
    except Exception as error:
        print(f"embedding_model_cpu_move_warning={model_key}: {error}")


def clear_embedding_memory(model_key: str) -> None:
    gc.collect()
    if not torch.cuda.is_available():
        return
    try:
        torch.cuda.synchronize(EMBEDDING_GPU_ID)
    except Exception:
        pass
    torch.cuda.empty_cache()
    try:
        torch.cuda.ipc_collect()
    except Exception:
        pass
    allocated = torch.cuda.memory_allocated(EMBEDDING_GPU_ID) / 1024**3
    reserved = torch.cuda.memory_reserved(EMBEDDING_GPU_ID) / 1024**3
    print(
        f"released_embedding_model={model_key} "
        f"cuda_allocated_gb={allocated:.3f} cuda_reserved_gb={reserved:.3f}"
    )

## Record Construction

In [ ]:
PAIR_AGGREGATION = "all_prompt_pairs"
GLOBAL_BANK_SAMPLING = "nested_fixed_seed_permutation"
PAIR_SPECS = (
    ("p1-p2", "edit", "generality", "edit - generality"),
    ("p1-p3", "edit", "locality", "edit - locality"),
    ("p2-p3", "generality", "locality", "generality - locality"),
)


def load_similarity_dataset(dataset_source: str):
    if dataset_source == "local":
        return load_from_disk(str(LOCAL_DATASET_PATH))
    if dataset_source == "huggingface":
        return load_dataset(DATASET_REPO, DATASET_CONFIG, split=DATASET_SPLIT)
    raise ValueError("dataset_source must be 'local' or 'huggingface'.")


def source_record_count(dataset, record_limit: int | None) -> int:
    if record_limit is None:
        return len(dataset)
    if int(record_limit) <= 0:
        raise ValueError("record_limit must be positive when set.")
    return min(int(record_limit), len(dataset))


def build_local_similarity_records(
    dataset,
    record_limit: int | None,
) -> list[dict[str, Any]]:
    records = []
    for source_index in range(source_record_count(dataset, record_limit)):
        source_record = dataset[source_index]
        role_prompts = {
            "edit": [source_record["edit"]["prompt"]],
            "generality": list(source_record["generality"]),
            "locality": [item["prompt"] for item in source_record["locality"]],
        }
        for role, prompts in role_prompts.items():
            if not prompts or any(not prompt for prompt in prompts):
                raise ValueError(f"Empty {role} prompt list at source index {source_index}.")
        records.append({"source_index": source_index, "role_prompts": role_prompts})
    return records


def build_global_similarity_records(
    dataset,
    record_limit: int | None,
) -> list[dict[str, Any]]:
    records = []
    for source_index in range(source_record_count(dataset, record_limit)):
        source_record = dataset[source_index]
        edit_prompt = source_record["edit"]["prompt"]
        generality_prompts = list(source_record["generality"])
        locality_prompts = [item["prompt"] for item in source_record["locality"]]
        if not edit_prompt or not generality_prompts or not locality_prompts:
            raise ValueError(f"Incomplete retrieval record at source index {source_index}.")
        records.append({
            "source_index": source_index,
            "edit_prompt": edit_prompt,
            "generality_prompts": generality_prompts,
            "locality_prompts": locality_prompts,
        })
    return records


def prompt_key(prompt: str) -> str:
    normalized = unicodedata.normalize("NFKC", prompt).casefold()
    characters = [
        character
        if character.isalnum() or unicodedata.category(character).startswith("S")
        else " "
        for character in normalized
    ]
    return " ".join("".join(characters).split())


def seeded_rng(random_seed: int, *parts) -> np.random.Generator:
    payload = "|".join(str(part) for part in (random_seed, *parts)).encode("utf-8")
    seed = int.from_bytes(hashlib.sha256(payload).digest()[:8], "little")
    return np.random.default_rng(seed)


def sample_values(values: list, limit: int | None, rng: np.random.Generator) -> list:
    if limit is None or len(values) <= limit:
        return values
    indices = np.sort(rng.choice(len(values), size=limit, replace=False))
    return [values[int(index)] for index in indices]


def resolve_global_bank_sizes(
    bank_sizes: list[int | None],
    universe_size: int,
) -> list[tuple[int | None, int]]:
    resolved = []
    seen = set()
    for requested_size in bank_sizes:
        actual_size = universe_size if requested_size is None else int(requested_size)
        if actual_size < 2:
            raise ValueError("Every global bank must contain at least two edits.")
        if actual_size > universe_size:
            print(f"skipped_bank_size={actual_size} universe_size={universe_size}")
            continue
        key = (requested_size is None, actual_size)
        if key not in seen:
            seen.add(key)
            resolved.append((requested_size, actual_size))
    if not resolved:
        raise ValueError("No configured global bank size fits the dataset.")
    return resolved

## Similarity Calculation

In [ ]:
def analyze_local_record_batch(
    record_batch: list[dict[str, Any]],
    model,
    model_key: str,
) -> list[dict[str, Any]]:
    prompts = [
        prompt
        for record in record_batch
        for role_prompts in record["role_prompts"].values()
        for prompt in role_prompts
    ]
    embeddings = encode_prompts(model, model_key, prompts, prompt_role="symmetric")

    analyzed_records = []
    offset = 0
    for record in record_batch:
        role_embeddings = {}
        prompt_counts = {}
        for role, role_prompts in record["role_prompts"].items():
            count = len(role_prompts)
            role_embeddings[role] = embeddings[offset : offset + count]
            prompt_counts[role] = count
            offset += count

        pair_values = {}
        for pair_key, left_role, right_role, _ in PAIR_SPECS:
            matrix = role_embeddings[left_role] @ role_embeddings[right_role].T
            pair_values[pair_key] = [
                round(float(value), 8) for value in matrix.reshape(-1)
            ]
        analyzed_records.append({
            "source_index": record["source_index"],
            "prompt_counts": prompt_counts,
            "pair_values": pair_values,
        })
    return analyzed_records


def aggregate_local_records(records: list[dict[str, Any]]) -> dict[str, Any]:
    if not records:
        raise ValueError("No records were analyzed.")

    pairs = {}
    for pair_key, _, _, label in PAIR_SPECS:
        values = np.asarray(
            [
                value
                for record in records
                for value in record["pair_values"][pair_key]
            ],
            dtype=np.float64,
        )
        if values.size == 0:
            raise ValueError(f"No values were produced for pair {pair_key}.")
        pairs[pair_key] = {
            "label": label,
            "count": int(values.size),
            "mean": round(float(values.mean()), 8),
            "std": round(float(values.std(ddof=0)), 8),
        }
    return {"record_count": len(records), "pairs": pairs}


def write_json_atomic(payload: dict, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(f"{destination.suffix}.tmp")
    with open(temporary, "w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, ensure_ascii=True, allow_nan=False)
    temporary.replace(destination)


def similarity_dataset_metadata(dataset, dataset_source: str) -> dict[str, Any]:
    metadata = {
        "source": dataset_source,
        "rows": len(dataset),
        "fingerprint": getattr(dataset, "_fingerprint", None),
    }
    if dataset_source == "local":
        metadata["dataset_id"] = "uniedit/final"
    else:
        metadata.update({
            "repo": DATASET_REPO,
            "config": DATASET_CONFIG,
            "split": DATASET_SPLIT,
        })
    return metadata


def run_local_similarity_analysis(
    dataset,
    records: list[dict[str, Any]],
    model,
    model_key: str,
    output_path: Path,
    record_batch_size: int,
    dataset_metadata: dict[str, Any],
    experiment: dict[str, Any],
) -> None:
    if record_batch_size <= 0:
        raise ValueError("record_batch_size must be positive.")

    completed_records = []
    for start in tqdm(
        range(0, len(records), record_batch_size),
        desc=f"Local embeddings [{model_key}]",
    ):
        completed_records.extend(
            analyze_local_record_batch(
                records[start : start + record_batch_size],
                model=model,
                model_key=model_key,
            )
        )

    completed_records.sort(key=lambda record: record["source_index"])
    result = build_result_header(
        model=model,
        model_key=model_key,
        dataset_metadata=dataset_metadata,
        experiment=experiment,
    )
    result.update({
        "pair_aggregation": PAIR_AGGREGATION,
        "pair_order": [
            {"pair": pair_key, "label": label}
            for pair_key, _, _, label in PAIR_SPECS
        ],
        "records": completed_records,
        "aggregate": aggregate_local_records(completed_records),
    })
    write_json_atomic(result, output_path)
    print(f"saved={output_path} records={len(completed_records)}")


def exact_positive_retrieval(
    query_embeddings: np.ndarray,
    bank: torch.Tensor,
    owner_positions: np.ndarray,
    retrieval_batch_size: int,
) -> dict[str, np.ndarray]:
    if len(query_embeddings) != len(owner_positions):
        raise ValueError("Each positive query must have one owner position.")
    device = bank.device
    ranks = []
    max_scores = []
    correct_scores = []
    best_negative_scores = []
    top1_correct = []
    with torch.inference_mode():
        for start in range(0, len(query_embeddings), retrieval_batch_size):
            stop = min(start + retrieval_batch_size, len(query_embeddings))
            queries = torch.as_tensor(
                query_embeddings[start:stop],
                dtype=torch.float32,
                device=device,
            )
            owners = torch.as_tensor(
                owner_positions[start:stop],
                dtype=torch.long,
                device=device,
            )
            scores = queries @ bank.T
            rows = torch.arange(len(owners), device=device)
            correct = scores[rows, owners]
            top_scores, top_indices = torch.topk(scores, k=2, dim=1)
            correct_is_top1 = top_indices[:, 0].eq(owners)
            best_negative = torch.where(
                correct_is_top1,
                top_scores[:, 1],
                top_scores[:, 0],
            )
            ranks.append((scores > correct[:, None]).sum(dim=1).add(1).cpu().numpy())
            max_scores.append(top_scores[:, 0].cpu().numpy())
            correct_scores.append(correct.cpu().numpy())
            best_negative_scores.append(best_negative.cpu().numpy())
            top1_correct.append(correct_is_top1.cpu().numpy())
            del queries, owners, scores, correct, top_scores, top_indices, best_negative
    return {
        "ranks": np.concatenate(ranks),
        "max_scores": np.concatenate(max_scores),
        "correct_scores": np.concatenate(correct_scores),
        "best_negative_scores": np.concatenate(best_negative_scores),
        "top1_correct": np.concatenate(top1_correct),
    }


def exact_locality_retrieval(
    query_embeddings: np.ndarray,
    bank: torch.Tensor,
    retrieval_batch_size: int,
) -> np.ndarray:
    device = bank.device
    maxima = []
    with torch.inference_mode():
        for start in range(0, len(query_embeddings), retrieval_batch_size):
            queries = torch.as_tensor(
                query_embeddings[start : start + retrieval_batch_size],
                dtype=torch.float32,
                device=device,
            )
            maxima.append((queries @ bank.T).max(dim=1).values.cpu().numpy())
            del queries
    return np.concatenate(maxima)


def calibrated_threshold(scores: np.ndarray, target_recall: float) -> float:
    if not 0.0 < target_recall <= 1.0:
        raise ValueError("global_target_recall must be in (0, 1].")
    return float(np.quantile(scores, 1.0 - target_recall, method="lower"))


def aggregate_repeat_metrics(repeats: list[dict[str, Any]]) -> dict[str, Any]:
    metric_names = [
        "recall_at_1",
        "recall_at_5",
        "recall_at_10",
        "mrr",
        "correct_margin_mean",
        "correct_margin_std",
        "threshold",
        "positive_activation_rate",
        "correct_routed_rate",
        "locality_false_activation_rate",
        "locality_max_score_mean",
        "locality_max_score_p95",
    ]
    aggregate = {}
    for metric_name in metric_names:
        values = np.asarray([repeat[metric_name] for repeat in repeats], dtype=np.float64)
        aggregate[metric_name] = {
            "mean": round(float(values.mean()), 8),
            "std": round(float(values.std(ddof=0)), 8),
        }
    return aggregate


def run_global_similarity_analysis(
    records: list[dict[str, Any]],
    model,
    model_key: str,
    output_path: Path,
    dataset_metadata: dict[str, Any],
    experiment: dict[str, Any],
) -> None:
    settings = experiment["settings"]
    edit_embeddings = encode_prompts(
        model,
        model_key,
        [record["edit_prompt"] for record in records],
        prompt_role="passage",
    )
    resolved_sizes = resolve_global_bank_sizes(settings["bank_sizes"], len(records))
    scales = []
    progress_total = len(resolved_sizes) * settings["repeats"]
    progress = tqdm(total=progress_total, desc=f"Global retrieval [{model_key}]")
    for requested_size, bank_size in resolved_sizes:
        repeat_results = []
        for repeat_index in range(settings["repeats"]):
            bank_rng = seeded_rng(settings["random_seed"], "bank", repeat_index)
            query_rng = seeded_rng(
                settings["random_seed"], "queries", bank_size, repeat_index
            )
            if bank_size == len(records):
                bank_indices = np.arange(len(records), dtype=np.int64)
            else:
                bank_indices = np.sort(
                    bank_rng.permutation(len(records))[:bank_size]
                )
            calibration_count = int(round(bank_size * settings["calibration_fraction"]))
            calibration_count = min(max(calibration_count, 1), bank_size - 1)
            shuffled_positions = query_rng.permutation(bank_size)
            calibration_positions = set(
                int(position) for position in shuffled_positions[:calibration_count]
            )
            evaluation_positions = [
                position for position in range(bank_size)
                if position not in calibration_positions
            ]

            calibration_queries = []
            evaluation_queries = []
            for owner_position, source_index in enumerate(bank_indices):
                destination = (
                    calibration_queries
                    if owner_position in calibration_positions
                    else evaluation_queries
                )
                destination.extend(
                    (prompt, owner_position)
                    for prompt in records[int(source_index)]["generality_prompts"]
                )
            total_limit = settings["positive_query_limit"]
            calibration_limit = max(1, int(round(total_limit * settings["calibration_fraction"])))
            evaluation_limit = max(1, total_limit - calibration_limit)
            calibration_queries = sample_values(
                calibration_queries, calibration_limit, query_rng
            )
            evaluation_queries = sample_values(
                evaluation_queries, evaluation_limit, query_rng
            )

            locality_by_key = {}
            for owner_position in evaluation_positions:
                source_index = int(bank_indices[owner_position])
                for prompt in records[source_index]["locality_prompts"]:
                    locality_by_key.setdefault(prompt_key(prompt), prompt)
            locality_queries = sample_values(
                list(locality_by_key.values()),
                settings["locality_query_limit"],
                query_rng,
            )
            if not calibration_queries or not evaluation_queries or not locality_queries:
                raise ValueError(
                    f"Insufficient global queries for bank_size={bank_size}, "
                    f"repeat={repeat_index}."
                )

            all_queries = (
                [prompt for prompt, _ in calibration_queries]
                + [prompt for prompt, _ in evaluation_queries]
                + locality_queries
            )
            query_embeddings = encode_prompts(
                model,
                model_key,
                all_queries,
                prompt_role="query",
            )
            calibration_stop = len(calibration_queries)
            evaluation_stop = calibration_stop + len(evaluation_queries)
            bank_embeddings = edit_embeddings[bank_indices]
            retrieval_bank = torch.as_tensor(
                bank_embeddings,
                dtype=torch.float32,
                device=torch.device(embedding_device()),
            )
            calibration = exact_positive_retrieval(
                query_embeddings[:calibration_stop],
                retrieval_bank,
                np.asarray([owner for _, owner in calibration_queries]),
                settings["retrieval_batch_size"],
            )
            evaluation = exact_positive_retrieval(
                query_embeddings[calibration_stop:evaluation_stop],
                retrieval_bank,
                np.asarray([owner for _, owner in evaluation_queries]),
                settings["retrieval_batch_size"],
            )
            locality_max_scores = exact_locality_retrieval(
                query_embeddings[evaluation_stop:],
                retrieval_bank,
                settings["retrieval_batch_size"],
            )
            threshold = calibrated_threshold(
                calibration["max_scores"],
                settings["target_recall"],
            )
            positive_active = evaluation["max_scores"] >= threshold
            locality_active = locality_max_scores >= threshold
            margins = evaluation["correct_scores"] - evaluation["best_negative_scores"]
            repeat_results.append({
                "repeat": repeat_index,
                "bank_size": bank_size,
                "calibration_owner_count": calibration_count,
                "calibration_query_count": len(calibration_queries),
                "positive_query_count": len(evaluation_queries),
                "locality_query_count": len(locality_queries),
                "recall_at_1": round(float(np.mean(evaluation["ranks"] <= 1)), 8),
                "recall_at_5": round(float(np.mean(evaluation["ranks"] <= 5)), 8),
                "recall_at_10": round(float(np.mean(evaluation["ranks"] <= 10)), 8),
                "mrr": round(float(np.mean(1.0 / evaluation["ranks"])), 8),
                "correct_margin_mean": round(float(margins.mean()), 8),
                "correct_margin_std": round(float(margins.std(ddof=0)), 8),
                "threshold": round(threshold, 8),
                "positive_activation_rate": round(float(positive_active.mean()), 8),
                "correct_routed_rate": round(
                    float(np.mean(positive_active & evaluation["top1_correct"])), 8
                ),
                "locality_false_activation_rate": round(float(locality_active.mean()), 8),
                "locality_max_score_mean": round(float(locality_max_scores.mean()), 8),
                "locality_max_score_p95": round(
                    float(np.quantile(locality_max_scores, 0.95)), 8
                ),
            })
            del query_embeddings, bank_embeddings, retrieval_bank
            progress.update(1)
        scales.append({
            "requested_bank_size": requested_size,
            "bank_size": bank_size,
            "repeats": repeat_results,
            "aggregate": aggregate_repeat_metrics(repeat_results),
        })
    progress.close()

    result = build_result_header(
        model=model,
        model_key=model_key,
        dataset_metadata=dataset_metadata,
        experiment=experiment,
    )
    result.update({
        "retrieval": "exact_blockwise_cosine",
        "retrieval_dtype": "float32",
        "scales": scales,
    })
    write_json_atomic(result, output_path)
    print(f"saved={output_path} scales={len(scales)}")

In [ ]:
def experiment_configuration(
    experiment_mode: str,
    record_limit: int | None,
    record_batch_size: int,
    global_bank_sizes: list[int | None],
    global_repeats: int,
    global_random_seed: int,
    global_positive_query_limit: int,
    global_locality_query_limit: int,
    global_calibration_fraction: float,
    global_target_recall: float,
    global_retrieval_batch_size: int,
) -> dict[str, Any]:
    experiment_mode = validate_experiment_mode(experiment_mode)
    if experiment_mode == "local":
        settings = {
            "pair_aggregation": PAIR_AGGREGATION,
            "record_batch_size": record_batch_size,
        }
    else:
        if global_repeats <= 0:
            raise ValueError("global_repeats must be positive.")
        if global_positive_query_limit < 2 or global_locality_query_limit <= 0:
            raise ValueError("Global query limits must be positive and usable.")
        if not 0.0 < global_calibration_fraction < 1.0:
            raise ValueError("global_calibration_fraction must be in (0, 1).")
        if global_retrieval_batch_size <= 0:
            raise ValueError("global_retrieval_batch_size must be positive.")
        settings = {
            "bank_sizes": global_bank_sizes,
            "bank_sampling": GLOBAL_BANK_SAMPLING,
            "repeats": global_repeats,
            "random_seed": global_random_seed,
            "positive_query_limit": global_positive_query_limit,
            "locality_query_limit": global_locality_query_limit,
            "calibration_fraction": global_calibration_fraction,
            "target_recall": global_target_recall,
            "retrieval_batch_size": global_retrieval_batch_size,
        }
    return {
        "mode": experiment_mode,
        "record_limit": record_limit,
        "settings": settings,
    }


def build_result_header(
    model,
    model_key: str,
    dataset_metadata: dict[str, Any],
    experiment: dict[str, Any],
) -> dict[str, Any]:
    spec = EMBEDDING_MODEL_SPECS[model_key]
    return {
        "embedding_model_key": model_key,
        "embedding_model": spec["repo"],
        "embedding_backend": spec["backend"],
        "input_prefixes": spec["prefixes"],
        "representation": REPRESENTATION_TYPE,
        "use_fp16": EMBEDDING_USE_FP16,
        "compute_dtype": embedding_compute_dtype(),
        "max_length_cap": EMBEDDING_MAX_LENGTH,
        "effective_max_length": embedding_effective_max_length(model, model_key),
        "dataset": dataset_metadata,
        "experiment": experiment,
    }


def result_matches_configuration(
    output_path: Path,
    model_key: str,
    experiment: dict[str, Any],
    dataset_metadata: dict[str, Any] | None = None,
) -> bool:
    if not output_path.exists():
        return False
    with open(output_path, encoding="utf-8") as file:
        result = json.load(file)
    spec = EMBEDDING_MODEL_SPECS[model_key]
    return (
        result.get("embedding_model_key") == model_key
        and result.get("embedding_model") == spec["repo"]
        and result.get("embedding_backend") == spec["backend"]
        and result.get("input_prefixes") == spec["prefixes"]
        and result.get("representation") == REPRESENTATION_TYPE
        and result.get("use_fp16") == EMBEDDING_USE_FP16
        and result.get("compute_dtype") == embedding_compute_dtype()
        and result.get("max_length_cap") == EMBEDDING_MAX_LENGTH
        and result.get("experiment") == experiment
        and (dataset_metadata is None or result.get("dataset") == dataset_metadata)
    )


def calculate_similarity(
    embedding_model_keys: list[str],
    experiment_mode: str,
    dataset_source: str,
    record_limit: int | None,
    record_batch_size: int,
    overwrite_output: bool,
    global_bank_sizes: list[int | None],
    global_repeats: int,
    global_random_seed: int,
    global_positive_query_limit: int,
    global_locality_query_limit: int,
    global_calibration_fraction: float,
    global_target_recall: float,
    global_retrieval_batch_size: int,
) -> None:
    model_keys = resolve_embedding_model_keys(embedding_model_keys)
    experiment = experiment_configuration(
        experiment_mode=experiment_mode,
        record_limit=record_limit,
        record_batch_size=record_batch_size,
        global_bank_sizes=global_bank_sizes,
        global_repeats=global_repeats,
        global_random_seed=global_random_seed,
        global_positive_query_limit=global_positive_query_limit,
        global_locality_query_limit=global_locality_query_limit,
        global_calibration_fraction=global_calibration_fraction,
        global_target_recall=global_target_recall,
        global_retrieval_batch_size=global_retrieval_batch_size,
    )
    dataset = load_similarity_dataset(dataset_source)
    dataset_metadata = similarity_dataset_metadata(dataset, dataset_source)
    calculation_keys = []
    for model_key in model_keys:
        output_path = similarity_output_path(model_key, experiment_mode, record_limit)
        if output_path.exists() and overwrite_output:
            output_path.unlink()
            print(f"removed_for_overwrite={output_path}")
        elif output_path.exists():
            if result_matches_configuration(
                output_path,
                model_key,
                experiment,
                dataset_metadata,
            ):
                print(f"skipped_existing={output_path}")
                continue
            print(f"recalculating_stale={output_path}")
        calculation_keys.append(model_key)
    if not calculation_keys:
        dataset = None
        gc.collect()
        return

    if experiment_mode == "local":
        records = build_local_similarity_records(dataset, record_limit)
    else:
        records = build_global_similarity_records(dataset, record_limit)
    print(
        f"experiment={experiment_mode} source={dataset_source} "
        f"records={len(records)} models={calculation_keys}"
    )
    failures = []
    for model_key in calculation_keys:
        model = None
        try:
            model = load_embedding_model(model_key)
            output_path = similarity_output_path(model_key, experiment_mode, record_limit)
            if experiment_mode == "local":
                run_local_similarity_analysis(
                    dataset,
                    records,
                    model=model,
                    model_key=model_key,
                    output_path=output_path,
                    record_batch_size=record_batch_size,
                    dataset_metadata=dataset_metadata,
                    experiment=experiment,
                )
            else:
                run_global_similarity_analysis(
                    records,
                    model=model,
                    model_key=model_key,
                    output_path=output_path,
                    dataset_metadata=dataset_metadata,
                    experiment=experiment,
                )
        except Exception as error:
            failures.append(f"{model_key}: {type(error).__name__}: {error}")
            print(f"embedding_model_failed={failures[-1]}")
        finally:
            if model is not None:
                move_embedding_model_to_cpu(model, model_key)
            model = None
            clear_embedding_memory(model_key)

    dataset = None
    records = None
    gc.collect()
    if failures:
        raise RuntimeError("Embedding models failed:\n" + "\n".join(failures))

## Tables And Plots

In [ ]:
PAIR_COLORS = {
    "p1-p2": "#0072B2",
    "p1-p3": "#E69F00",
    "p2-p3": "#009E73",
}


def aggregate_pair_table(results: list[tuple[str, dict]], show_std: bool) -> pd.DataFrame:
    rows = []
    for model_key, result in results:
        row = {"Embedding model": model_key}
        for pair_key, _, _, label in PAIR_SPECS:
            pair = result["aggregate"]["pairs"][pair_key]
            row[label] = (
                f"{pair['mean']:.4f} +/- {pair['std']:.4f}"
                if show_std
                else f"{pair['mean']:.4f}"
            )
        rows.append(row)
    return pd.DataFrame(rows).set_index("Embedding model")


def observation_count_table(results: list[tuple[str, dict]]) -> pd.DataFrame:
    rows = []
    for model_key, result in results:
        row = {
            "Embedding model": model_key,
            "Records": result["aggregate"]["record_count"],
        }
        for pair_key, _, _, label in PAIR_SPECS:
            row[label] = result["aggregate"]["pairs"][pair_key]["count"]
        rows.append(row)
    return pd.DataFrame(rows).set_index("Embedding model")


def summarize_pairwise_preference(result: dict) -> dict:
    tolerance = 1e-12
    gaps = np.concatenate([
        (
            np.asarray(record["pair_values"]["p1-p2"], dtype=np.float64)[:, None]
            - np.asarray(record["pair_values"]["p1-p3"], dtype=np.float64)[None, :]
        ).reshape(-1)
        for record in result["records"]
    ])
    generality_preferred = int(np.count_nonzero(gaps > tolerance))
    locality_preferred = int(np.count_nonzero(gaps < -tolerance))
    ties = int(gaps.size - generality_preferred - locality_preferred)
    return {
        "comparisons": int(gaps.size),
        "mean_gap": float(gaps.mean()),
        "gap_std": float(gaps.std(ddof=0)),
        "generality_preferred": generality_preferred / gaps.size,
        "locality_preferred": locality_preferred / gaps.size,
        "ties": ties / gaps.size,
    }


def pairwise_preference_table(results: list[tuple[str, dict]]) -> pd.DataFrame:
    rows = []
    for model_key, result in results:
        preference = result["pairwise_preference"]
        rows.append({
            "Embedding model": model_key,
            "Comparisons": preference["comparisons"],
            "Mean gap": preference["mean_gap"],
            "Gap std": preference["gap_std"],
            "Generality preferred": preference["generality_preferred"],
            "Locality preferred": preference["locality_preferred"],
            "Ties": preference["ties"],
        })
    return pd.DataFrame(rows).set_index("Embedding model")


def plot_local_similarity(results: list[tuple[str, dict]], show_std: bool) -> None:
    model_keys = [model_key for model_key, _ in results]
    pair_specs = list(PAIR_SPECS)
    positions = np.arange(len(model_keys), dtype=float)
    group_width = 0.78
    bar_width = group_width / len(pair_specs)
    offsets = (
        np.arange(len(pair_specs), dtype=float) - (len(pair_specs) - 1) / 2
    ) * bar_width

    all_lower = []
    all_upper = []
    figure, axis = plt.subplots(figsize=(18, 7))
    for pair_offset, (pair_key, _, _, label) in zip(offsets, pair_specs):
        means = np.asarray([
            result["aggregate"]["pairs"][pair_key]["mean"]
            for _, result in results
        ])
        stds = np.asarray([
            result["aggregate"]["pairs"][pair_key]["std"]
            for _, result in results
        ])
        axis.bar(
            positions + pair_offset,
            means,
            width=bar_width * 0.92,
            yerr=stds if show_std else None,
            label=f"{pair_key}: {label}",
            color=PAIR_COLORS[pair_key],
            capsize=5 if show_std else 0,
            error_kw={"elinewidth": 1.25, "capthick": 1.25},
        )
        all_lower.extend((means - stds if show_std else means).tolist())
        all_upper.extend((means + stds if show_std else means).tolist())

    y_min = min(0.0, min(all_lower))
    y_max = max(all_upper)
    padding = max(0.05, (y_max - y_min) * 0.12)
    axis.set_ylim(
        max(-1.0, y_min - (padding if y_min < 0 else 0)),
        min(1.0, y_max + padding),
    )
    axis.yaxis.set_major_locator(MaxNLocator(nbins=10))
    axis.set_xticks(positions, model_keys, rotation=20, ha="right")
    axis.set_ylabel("cosine similarity")
    record_counts = {result["aggregate"]["record_count"] for _, result in results}
    count_label = str(record_counts.pop()) if len(record_counts) == 1 else "mixed"
    axis.set_title(f"Local prompt similarity | n={count_label}", fontweight="bold")
    axis.grid(axis="y", alpha=0.22)
    axis.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), title="prompt pair")
    figure.tight_layout(rect=(0, 0, 0.82, 1))
    plt.show()


def global_result_table(results: list[tuple[str, dict]], group: str) -> pd.DataFrame:
    metric_groups = {
        "retrieval": [
            ("recall_at_1", "Recall@1"),
            ("recall_at_5", "Recall@5"),
            ("recall_at_10", "Recall@10"),
            ("mrr", "MRR"),
            ("correct_margin_mean", "Correct margin"),
        ],
        "routing": [
            ("threshold", "Threshold"),
            ("positive_activation_rate", "Positive activation"),
            ("correct_routed_rate", "Correct routed"),
            ("locality_false_activation_rate", "Locality false activation"),
            ("locality_max_score_p95", "Locality max p95"),
        ],
    }
    rows = []
    for model_key, result in results:
        for scale in result["scales"]:
            row = {
                "Embedding model": model_key,
                "Bank size": scale["bank_size"],
                "Repeats": len(scale["repeats"]),
            }
            for metric_key, label in metric_groups[group]:
                metric = scale["aggregate"][metric_key]
                row[label] = f"{metric['mean']:.4f} +/- {metric['std']:.4f}"
            rows.append(row)
    return pd.DataFrame(rows).set_index(["Embedding model", "Bank size"])


def global_query_count_table(results: list[tuple[str, dict]]) -> pd.DataFrame:
    rows = []
    for model_key, result in results:
        for scale in result["scales"]:
            repeats = scale["repeats"]
            rows.append({
                "Embedding model": model_key,
                "Bank size": scale["bank_size"],
                "Calibration queries": int(np.mean([
                    repeat["calibration_query_count"] for repeat in repeats
                ])),
                "Positive queries": int(np.mean([
                    repeat["positive_query_count"] for repeat in repeats
                ])),
                "Locality queries": int(np.mean([
                    repeat["locality_query_count"] for repeat in repeats
                ])),
            })
    return pd.DataFrame(rows).set_index(["Embedding model", "Bank size"])


def plot_global_similarity(results: list[tuple[str, dict]]) -> None:
    plot_specs = [
        ("recall_at_1", "Recall@1"),
        ("mrr", "MRR"),
        ("correct_routed_rate", "Correct routed rate"),
        ("locality_false_activation_rate", "Locality false activation rate"),
    ]
    figure, axes = plt.subplots(2, 2, figsize=(18, 11), sharex=True)
    for axis, (metric_key, label) in zip(axes.flat, plot_specs):
        for model_key, result in results:
            sizes = np.asarray([scale["bank_size"] for scale in result["scales"]])
            means = np.asarray([
                scale["aggregate"][metric_key]["mean"] for scale in result["scales"]
            ])
            stds = np.asarray([
                scale["aggregate"][metric_key]["std"] for scale in result["scales"]
            ])
            axis.errorbar(
                sizes,
                means,
                yerr=stds,
                marker="o",
                linewidth=2,
                capsize=4,
                label=model_key,
            )
        axis.set_xscale("log")
        axis.set_ylabel(label)
        axis.grid(alpha=0.22)
        axis.set_ylim(bottom=0.0, top=1.02)
    for axis in axes[-1]:
        axis.set_xlabel("edit-bank size (log scale)")
    axes[0, 1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
    figure.suptitle("Global edit retrieval and abstention", fontweight="bold")
    figure.tight_layout(rect=(0, 0, 0.88, 0.97))
    plt.show()


def visualize_similarity(
    embedding_model_keys: list[str],
    experiment_mode: str,
    record_limit: int | None,
    show_std: bool,
    experiment: dict[str, Any],
) -> None:
    model_keys = resolve_embedding_model_keys(embedding_model_keys)
    results = []
    for model_key in model_keys:
        output_path = similarity_output_path(model_key, experiment_mode, record_limit)
        if not output_path.exists():
            raise FileNotFoundError(
                f"Embedding result does not exist: {output_path}. Calculate it first."
            )
        if not result_matches_configuration(output_path, model_key, experiment):
            raise ValueError(f"Stale embedding result: {output_path}. Recalculate it.")
        with open(output_path, encoding="utf-8") as file:
            result = json.load(file)
        if experiment_mode == "local":
            result["pairwise_preference"] = summarize_pairwise_preference(result)
            result.pop("records", None)
        results.append((model_key, result))

    if experiment_mode == "local":
        display(aggregate_pair_table(results, show_std=show_std).style)
        display(
            pairwise_preference_table(results).style.format({
                "Comparisons": "{:,}",
                "Mean gap": "{:.4f}",
                "Gap std": "{:.4f}",
                "Generality preferred": "{:.4f}",
                "Locality preferred": "{:.4f}",
                "Ties": "{:.4f}",
            })
        )
        display(observation_count_table(results).style.format("{:,}"))
        plot_local_similarity(results, show_std=show_std)
    else:
        display(global_result_table(results, "retrieval").style)
        display(global_result_table(results, "routing").style)
        display(global_query_count_table(results).style.format("{:,}"))
        plot_global_similarity(results)

## Experiment Configuration

In [ ]:
# Choose one experiment without duplicating the calculation/visualization pipeline.
EXPERIMENT_MODE = "local"  # "local" or "global"
DATASET_SOURCE = "local"  # Use "huggingface" after publishing the rebuilt artifact.

# Comment out a line to disable it; list order controls calculation and display order.
EMBEDDING_MODEL_KEYS = [
    "all-mpnet-base-v2",
    "all-MiniLM-L6-v2",
    "all-MiniLM-L12-v2",
    "e5-base-v2",
    "e5-large-v2",
    "bge-base-en-v1.5",
    "bge-large-en-v1.5",
    "bge-m3",
]

EMBEDDING_GPU_ID = 0
EMBEDDING_USE_FP16 = False
EMBEDDING_MAX_LENGTH = 384
EMBEDDING_BATCH_SIZE = 64
RECORD_BATCH_SIZE = 64  # Local mode only.
RECORD_LIMIT = None  # Set a positive integer for diagnostic result files.
OVERWRITE_EXISTING_RESULT = False

# Global retrieval scales; None means the complete available edit bank.
GLOBAL_BANK_SIZES = [
    10,
    100,
    1000,
    2000,
    5000,
    None,
]
GLOBAL_REPEATS = 3
GLOBAL_RANDOM_SEED = 42
GLOBAL_POSITIVE_QUERY_LIMIT = 2000
GLOBAL_LOCALITY_QUERY_LIMIT = 2000
GLOBAL_CALIBRATION_FRACTION = 0.2
GLOBAL_TARGET_RECALL = 0.95
GLOBAL_RETRIEVAL_BATCH_SIZE = 256

PLOT_SHOW_STD = True

EXPERIMENT = experiment_configuration(
    experiment_mode=EXPERIMENT_MODE,
    record_limit=RECORD_LIMIT,
    record_batch_size=RECORD_BATCH_SIZE,
    global_bank_sizes=GLOBAL_BANK_SIZES,
    global_repeats=GLOBAL_REPEATS,
    global_random_seed=GLOBAL_RANDOM_SEED,
    global_positive_query_limit=GLOBAL_POSITIVE_QUERY_LIMIT,
    global_locality_query_limit=GLOBAL_LOCALITY_QUERY_LIMIT,
    global_calibration_fraction=GLOBAL_CALIBRATION_FRACTION,
    global_target_recall=GLOBAL_TARGET_RECALL,
    global_retrieval_batch_size=GLOBAL_RETRIEVAL_BATCH_SIZE,
)

## Manual Actions

In [ ]:
# Calculate and save one JSON per enabled embedding model:
# calculate_similarity(
#     embedding_model_keys=EMBEDDING_MODEL_KEYS,
#     experiment_mode=EXPERIMENT_MODE,
#     dataset_source=DATASET_SOURCE,
#     record_limit=RECORD_LIMIT,
#     record_batch_size=RECORD_BATCH_SIZE,
#     overwrite_output=OVERWRITE_EXISTING_RESULT,
#     global_bank_sizes=GLOBAL_BANK_SIZES,
#     global_repeats=GLOBAL_REPEATS,
#     global_random_seed=GLOBAL_RANDOM_SEED,
#     global_positive_query_limit=GLOBAL_POSITIVE_QUERY_LIMIT,
#     global_locality_query_limit=GLOBAL_LOCALITY_QUERY_LIMIT,
#     global_calibration_fraction=GLOBAL_CALIBRATION_FRACTION,
#     global_target_recall=GLOBAL_TARGET_RECALL,
#     global_retrieval_batch_size=GLOBAL_RETRIEVAL_BATCH_SIZE,
# )

# Visualize enabled results in configured model and bank-size order:
# visualize_similarity(
#     embedding_model_keys=EMBEDDING_MODEL_KEYS,
#     experiment_mode=EXPERIMENT_MODE,
#     record_limit=RECORD_LIMIT,
#     show_std=PLOT_SHOW_STD,
#     experiment=EXPERIMENT,
# )